In [ ]:
%%capture
import os, sys
if 'google.colab' in sys.modules:
    !pip install unsloth
else:
    !pip install unsloth sacrebleu regex open-tamil

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face.")

In [ ]:
from unsloth import FastModel
import torch

SEED = 42
MAX_SEQ_LENGTH = 512

model, tokenizer = FastModel.from_pretrained(
    model_name     = "unsloth/gemma-3-270m-it",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
    load_in_8bit   = False,
    full_finetuning = False,
)

print(f"Model dtype : {model.dtype}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r             = 16,
    lora_alpha    = 16,
    lora_dropout  = 0,
    bias          = "none",
    random_state  = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
import pandas as pd, re, random
import numpy as np
from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/robinsavio/tamilvenba/final_dataset_1.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df)} rows, columns: {list(df.columns)}")

def preprocess_venba(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('்்', '்')
    return text.strip()

def preprocess_pathaseth(text):
    return re.sub(r'\s+', ' ', text).strip()

df['Venba_clean']      = df['Venba'].apply(preprocess_venba)
df['Pathavurai_clean'] = df['Pathavurai'].apply(preprocess_pathaseth)
df = df.drop_duplicates(subset=['Venba_clean', 'Pathavurai_clean']).reset_index(drop=True)
print(f"After dedup: {len(df)} rows")

train_df, test_df = train_test_split(
    df[['Venba_clean', 'Pathavurai_clean']],
    test_size=0.20, random_state=SEED, shuffle=True
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset
import regex, unicodedata

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

INSTRUCTION = (
    "கீழ்க்காணும் தமிழ் வெண்பாவை, அதன் சொற்களை சரியான எல்லைகளுடன் "
    "பிரித்து எழுதுக (பதப்பிரிவு). சொற்களின் வரிசை மாறாமல், "
    "ஒட்டிய சொற்களை பிரிக்கவும், தேவையெனில் பிரிந்த எழுத்துக்களை "
    "இணைத்து சரியான சொல்லை உருவாக்கவும். பதப்பிரிவை மட்டும் தரவும்."
)

PROBLEMATIC_AKSHARAS = {
    'யு', 'றா', 'ஞா', 'யே', 'யை', 'றே', 'கெ', 'றோ', 'யெ', 'யொ',
    'றெ', 'ழா', 'நூ', 'லெ', 'வொ', 'ரெ', 'றொ', 'கீ', 'டே', 'லொ',
    'னொ', 'டொ', 'ணே', 'ளெ', 'ணோ', 'ரொ', 'றூ', 'ணீ', 'றீ', 'ழீ',
    'னூ', 'ணெ', 'வூ', 'யீ', 'டீ', 'ளொ', 'லூ', 'நொ', 'ணொ', 'டூ',
    'நை', 'கௌ', 'ளூ', 'ழே', 'ழூ', 'ஞை', 'ளீ', 'ஜை', 'ணூ', 'ழொ',
    'ழெ', 'ஞு', 'ஞெ', 'ஞூ', 'ஸா', 'ஜூ', 'எா', 'எீ', 'ழோ', 'பௌ',
}

def clean_grapheme_split(text, tokenizer):
    """
    Split Tamil text into grapheme clusters.
    Uses | as intra-word separator, space as inter-word separator.
    Uses pre-computed lookup table for problematic aksharas instead
    of recursive merging — more reliable and handles all edge cases.
    """
    words = text.split(' ')
    result_words = []

    for word in words:
        if not word.strip():
            continue

        clusters = regex.findall(r'\X', word)
        clusters = [
            unicodedata.normalize('NFC', c) for c in clusters
            if c.strip() and any('\u0B80' <= ch <= '\u0BFF' for ch in c)
        ]

        if not clusters:
            result_words.append(word)
            continue

        merged = []
        i = 0
        while i < len(clusters):
            c = clusters[i]
            tokens = tokenizer.tokenize(c)
            if (c in PROBLEMATIC_AKSHARAS or len(tokens) > 1) and i + 1 < len(clusters):
                combined = c + clusters[i+1]
                while len(tokenizer.tokenize(combined)) > 1 and i + 2 < len(clusters):
                    i += 1
                    combined = combined + clusters[i+1]
                merged.append(combined)
                i += 2
            else:
                merged.append(c)
                i += 1

        result_words.append('|'.join(merged))

    return ' '.join(result_words)

def post_process_grapheme(text):
    """Remove | separators after generation to restore normal Tamil text."""
    return text.replace('|', '')

def format_as_conversations(df_split):
    """Convert df rows to conversations with grapheme-split Venba and Pathavurai."""
    data = []
    for _, row in df_split.iterrows():
        venba_g = clean_grapheme_split(row['Venba_clean'], tokenizer)
        pathu_g = clean_grapheme_split(row['Pathavurai_clean'], tokenizer)
        data.append({
            "conversations": [
                {"role": "user",      "content": f"{INSTRUCTION}\n\nவெண்பா: {venba_g}\nபதப்பிரிவு:"},
                {"role": "assistant", "content": pathu_g},
            ]
        })
    return Dataset.from_list(data)

print("Applying grapheme tokenization to dataset...")
train_dataset = format_as_conversations(train_df)
test_dataset  = format_as_conversations(test_df)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print()
print("=== Sample (train[0]) ===")
print(train_dataset[0]['conversations'][0]['content'])
print("---")
print(train_dataset[0]['conversations'][1]['content'])


In [ ]:
sample_venba = train_df.iloc[0]['Venba_clean']
sample_pathu = train_df.iloc[0]['Pathavurai_clean']

raw_tokens = tokenizer.tokenize(sample_venba)
print("=== RAW BPE TOKENS (current) ===")
print(f"Text  : {sample_venba}")
print(f"Tokens: {raw_tokens}")
print(f"Count : {len(raw_tokens)}")
print()

grapheme_text = clean_grapheme_split(sample_venba, tokenizer)
graph_tokens  = tokenizer.tokenize(grapheme_text)
print("=== GRAPHEME TOKENS (new) ===")
print(f"Text  : {grapheme_text}")
print(f"Tokens: {graph_tokens}")
print(f"Count : {len(graph_tokens)}")

In [ ]:
print("=" * 70)
print("GRAPHEME TOKENIZATION VERIFICATION — 5 REAL EXAMPLES")
print("=" * 70)

for i in range(5):
    row = train_df.iloc[i]
    venba = row['Venba_clean']
    pathu = row['Pathavurai_clean']

    venba_g = clean_grapheme_split(venba, tokenizer)
    pathu_g = clean_grapheme_split(pathu, tokenizer)

    all_units = venba_g.replace(' ', '|').split('|') + pathu_g.replace(' ', '|').split('|')
    splits_found = []
    for unit in all_units:
        if not unit.strip():
            continue
        tokens = tokenizer.tokenize(unit)
        if len(tokens) > 1:
            splits_found.append(f"{unit}→{tokens}")

    print(f"\n[{i+1}] ORIGINAL VENBA   : {venba}")
    print(f"    GRAPHEME VENBA   : {venba_g}")
    print(f"    ORIGINAL PATHU   : {pathu}")
    print(f"    GRAPHEME PATHU   : {pathu_g}")
    print(f"    ROUNDTRIP VENBA  : {post_process_grapheme(venba_g)}")
    print(f"    ROUNDTRIP MATCH  : {'✅' if post_process_grapheme(venba_g) == venba else '❌'}")
    print(f"    TOKEN CLEAN      : {'✅ All units = 1 token' if not splits_found else f'⚠️  {splits_found}'}")

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts  = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset  = test_dataset.map(formatting_prompts_func, batched=True)

print("=== Formatted text (train[0]) ===")
print(train_dataset[0]['text'])

encoded = tokenizer(train_dataset[0]['text'])['input_ids']
bos_count = sum(1 for t in encoded[:5] if t == tokenizer.bos_token_id)
print(f"\nBOS count in first 5 tokens: {bos_count} (should be 1)")
response_ids = tokenizer.encode("<start_of_turn>model\n", add_special_tokens=False)
found = any(encoded[i:i+len(response_ids)] == response_ids for i in range(len(encoded)-len(response_ids)+1))
print(f"Response template found: {found} (should be True)")

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb.init(
    project = "tamil-pathasetham-gemma3",
    name    = "gemma3-270m-grapheme-r16-a16-e5-8020-Changed",
    config  = {
        "model": "unsloth/gemma-3-270m-it",
        "lora_r": 16, "lora_alpha": 16, "lora_dropout": 0,
        "learning_rate": 2e-4, "lr_scheduler": "linear",
        "epochs": 5, "effective_batch_size": 8,
        "max_seq_length": MAX_SEQ_LENGTH,
        "train_samples": len(train_dataset),
        "test_samples": len(test_dataset),
    }
)
print("W&B run:", wandb.run.url)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

OUTPUT_DIR = "/kaggle/working/gemma3-grapheme"

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,

    args = SFTConfig(
        output_dir                    = OUTPUT_DIR,
        run_name                      = "gemma3-pathasetham-sft",

        num_train_epochs              = 5,
        per_device_train_batch_size   = 2,
        gradient_accumulation_steps   = 4,
        per_device_eval_batch_size    = 8,

        learning_rate                 = 2e-4,
        lr_scheduler_type             = "linear",
        warmup_ratio                  = 0.05,
        weight_decay                  = 0.01,

        fp16                          = not torch.cuda.is_bf16_supported(),
        bf16                          = torch.cuda.is_bf16_supported(),

        max_seq_length                = MAX_SEQ_LENGTH,
        dataset_text_field            = "text",
        packing                       = False,

        eval_strategy                 = "epoch",
        save_strategy                 = "epoch",
        save_total_limit              = 2,
        load_best_model_at_end        = True,
        metric_for_best_model         = "eval_loss",

        logging_steps                 = 10,
        report_to                     = "wandb",

        seed                          = SEED,
        data_seed                     = SEED,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

print(f"Effective batch size     : {2*4}")
print(f"Training steps per epoch : {len(train_dataset) // (2*4)}")
print(f"Total training steps     : {len(train_dataset) // (2*4) * 5}")

In [ ]:
sample     = trainer.train_dataset[0]
input_ids  = sample['input_ids']
labels     = sample['labels']

print(f"Total tokens  : {len(input_ids)}")
print(f"Masked (-100) : {sum(1 for l in labels if l == -100)}")
print(f"Unmasked      : {sum(1 for l in labels if l != -100)}")

unmasked_ids = [t for t, l in zip(input_ids, labels) if l != -100]
print(f"\nDecoded unmasked (response) tokens:")
print(tokenizer.decode(unmasked_ids))
print()
print("✅ Should show ONLY the Pathasetham text + <end_of_turn>")
print("❌ If it shows the instruction/venba text, masking is broken")

In [ ]:
trainer_stats = trainer.train()

print("\n=== Training Complete ===")
print(f"Runtime      : {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss   : {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_steps, train_losses, eval_epochs, eval_losses = [], [], [], []

for entry in log_history:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_epochs.append(entry['epoch'])
        eval_losses.append(entry['eval_loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_steps, train_losses, color='steelblue', linewidth=2)
axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, eval_losses, color='darkorange', linewidth=2, marker='o', markersize=8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss'); axes[1].set_xticks(eval_epochs)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Gemma 3 270M — Tamil Pathasetham SFT', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curves.png', dpi=150)
plt.show()
print("Saved: /kaggle/working/loss_curves.png")

In [ ]:
LORA_PATH = "/kaggle/working/gemma3-pathasetham-lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapter saved to: {LORA_PATH}")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

REPO_NAME = "Razon2006/gemma3-tamil-Grapheme-NewChange"
model.push_to_hub(REPO_NAME, token=hf_token)
tokenizer.push_to_hub(REPO_NAME, token=hf_token)
print(f"Pushed LoRA to: https://huggingface.co/{REPO_NAME}")

In [ ]:
FastModel.for_inference(model)

def format_inference_prompt(venba):
    venba_g = clean_grapheme_split(venba, tokenizer)
    messages = [{
        "role": "user",
        "content": f"{INSTRUCTION}\n\nவெண்பா: {venba_g}\nபதப்பிரிவு:"
    }]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_pathaseth(venba, max_new_tokens=256):
    prompt = format_inference_prompt(venba)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            use_cache      = True,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return post_process_grapheme(raw)

test_cases = [
    ("உடைமையு ளின்மை விருந்தோம்ப லோம்பா மடமை மடவார்க ணுண்டு",
     "உடைமையுள் இன்மை விருந்தோம்பல் ஓம்பா மடமை மடவார்கண் உண்டு"),
    ("வண்மைதரு மாகமநூன் வைத்த பொருள்வழுவா உண்மை விளக்க முரைசெய்யத்",
     "வண்மை தரும் ஆகம நூல் வைத்த பொருள் வழுவா உண்மை விளக்கம் உரை செய்யத்"),
    ("அகர முதல எழுத்தெல்லாம் ஆதி பகவன் முதற்றே உலகு", None),
]

print("=" * 60)
print("QUICK INFERENCE GLIMPSE")
print("=" * 60)
for i, (venba, expected) in enumerate(test_cases):
    pred = generate_pathaseth(venba)
    print(f"\n[{i+1}] INPUT   : {venba}")
    if expected:
        print(f"    EXPECTED: {expected}")
    print(f"    OUTPUT  : {pred}")

In [ ]:
import sacrebleu
from tqdm import tqdm

predictions = []
references  = list(test_df['Pathavurai_clean'])

print(f"Evaluating {len(test_df)} test examples (single-example generation)...")
print("Note: this takes a while — expected ~30-60 mins on T4.")

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = generate_pathaseth(row['Venba_clean'])
    predictions.append(pred)

exact_matches = sum(p.strip() == r.strip() for p, r in zip(predictions, references))
word_correct, word_total = 0, 0
for pred, ref in zip(predictions, references):
    pw, rw = pred.split(), ref.split()
    word_correct += sum(p == r for p, r in zip(pw, rw))
    word_total   += max(len(pw), len(rw))

bleu = sacrebleu.corpus_bleu(predictions, [references])
n    = len(test_df)

print("\n" + "="*50)
print(f"Exact Match Accuracy : {exact_matches}/{n} = {100*exact_matches/n:.2f}%")
print(f"BLEU-4 Score         : {bleu.score:.2f}")
print(f"Word-level Accuracy  : {100*word_correct/word_total:.2f}%")
print("="*50)

In [ ]:
import os

exact_matches_count = sum(p.strip() == r.strip() for p, r in zip(predictions, references))

wandb.log({
    "final/exact_match_accuracy": 100 * exact_matches_count / len(references),
    "final/bleu4": bleu.score,
    "final/word_level_accuracy": 100 * word_correct / word_total,
})

if os.path.exists('/kaggle/working/loss_curves.png'):
    wandb.log({"loss_curve": wandb.Image('/kaggle/working/loss_curves.png')})

table = wandb.Table(columns=["Venba", "Reference", "Predicted", "Exact Match"])
for i in range(min(20, len(predictions))):
    table.add_data(
        test_df.iloc[i]['Venba_clean'],
        references[i],
        predictions[i],
        predictions[i].strip() == references[i].strip()
    )
wandb.log({"sample_predictions": table})
print("Logged to W&B:", wandb.run.url)

In [ ]:
print("=" * 60)
print("QUALITATIVE EVALUATION (first 10 test examples)")
print("=" * 60)

for i in range(min(10, len(predictions))):
    pred = predictions[i]
    ref  = references[i]
    pw   = pred.split()
    rw   = ref.split()
    diff = []
    for p, r in zip(pw, rw):
        diff.append(f"✓{p}" if p == r else f"✗[{p}≠{r}]")
    is_exact = pred.strip() == ref.strip()
    print(f"\n[{i+1}] VENBA    : {test_df.iloc[i]['Venba_clean']}")
    print(f"    REF      : {ref}")
    print(f"    PRED     : {pred}")
    print(f"    DIFF     : {' '.join(diff)}")
    print(f"    EXACT    : {'✅' if is_exact else '❌'}")

In [ ]:
import pandas as pd

eval_df = test_df.copy().reset_index(drop=True)
eval_df['predicted']   = predictions
eval_df['exact_match'] = [
    p.strip() == r.strip() for p, r in zip(predictions, references)
]

EVAL_CSV = "/kaggle/working/evaluation_results.csv"
eval_df.to_csv(EVAL_CSV, index=False)
print(f"Saved evaluation results to: {EVAL_CSV}")
print(f"Exact match rate: {eval_df['exact_match'].mean()*100:.2f}%")

In [ ]:
wandb.finish()
print("W&B run closed.")